# Лабораторная работа №1. Сбор и анализ данных OpenAlex

**Цель работы:** собрать 1200 записей с веб-ресурса, сохранить их в SQLite, сделать возможность дозагрузки новых данных и выполнить не меньше 5 агрегирующих SQL-запросов через `pandas` и `SQLAlchemy`.

В качестве источника я выбрала **OpenAlex** открытый каталог научных публикаций. Данные получаю через публичный API `https://api.openalex.org/works`.

## 1. Подключение библиотек

Сначала я подключила библиотеки, которые понадобятся дальше: `requests` для запросов к OpenAlex, `pandas` для таблицы с данными и `SQLAlchemy` для работы с SQLite. `time` использую, чтобы делать небольшую паузу между запросами к API.

In [110]:
import requests
import pandas as pd
import time
from sqlalchemy import create_engine, text

## 2. Получение данных

OpenAlex отдает данные постранично, поэтому я запрашиваю по 100 работ за раз. Чтобы получить 1200 записей, нужно 12 страниц.

Я беру только те поля, которые дальше можно нормально анализировать: название, дату и год публикации, тип работы, число цитирований, авторов, основную тему, Open Access, язык и DOI.

Также я ограничила дату публикации **17.11.2026**, чтобы в выборку не попадали работы с будущими датами (В OpenAlex такое встречается из-за метаданных издателей)

In [111]:
API_URL = "https://api.openalex.org/works"
MAX_PUBLICATION_DATE = "2026-11-17"

FIELDS = ",".join([
    "id", "title", "publication_date", "publication_year", "type",
    "cited_by_count", "authorships", "primary_topic", "open_access",
    "language", "doi"
])

In [112]:
def collect_works(target_count=1200):
    collected_works = []
    per_page = 100
    pages = (target_count + per_page - 1) // per_page

    for page in range(1, pages + 1):
        params = {
            "per_page": per_page,
            "page": page,
            "filter": f"to_publication_date:{MAX_PUBLICATION_DATE}",
            "sort": "publication_date:desc",
            "select": FIELDS
        }

        response = requests.get(API_URL, params=params, timeout=60)
        response.raise_for_status()

        data = response.json()
        collected_works.extend(data["results"])
        print("Собрано:", len(collected_works))
        time.sleep(0.5)

    return collected_works[:target_count]

## 3. Преобразование ответа API

Некоторые поля OpenAlex приходят как вложенные структуры.
(`authorships` список авторов)

Чтобы потом не работать с вложенными словарями и списками, я в `parse_work()` сразу превращаю их в обычные столбцы. Из авторов беру первого автора и отдельно считаю общее число авторов. Из `open_access` оставляю признак открытого доступа и его статус

In [113]:
def parse_work(work):
    authorships = work.get("authorships") or []
    first_author = "Не указан"

    if authorships:
        author = authorships[0].get("author") or {}
        first_author = author.get("display_name") or "Не указан"

    topic = work.get("primary_topic") or {}
    open_access = work.get("open_access") or {}

    return {
        "work_id": work.get("id"),
        "title": work.get("title") or "Без названия",
        "publication_date": work.get("publication_date"),
        "publication_year": work.get("publication_year"),
        "type": work.get("type") or "unknown",
        "cited_by_count": work.get("cited_by_count", 0),
        "first_author": first_author,
        "authors_count": len(authorships),
        "primary_topic": topic.get("display_name") or "Не указана",
        "is_open_access": open_access.get("is_oa", False),
        "oa_status": open_access.get("oa_status") or "unknown",
        "language": work.get("language") or "unknown",
        "doi": work.get("doi") or ""
    }

## 4. Сбор данных и создание DataFrame

После этого я собираю 1200 последних публикаций, преобразую каждую через `parse_work()` и создаю `DataFrame`.

Одна строка таблицы одна научная работа. В итоге у меня получается 1200 строк и 13 признаков

In [114]:
raw_works = collect_works(1200)

parsed_rows = []
for work in raw_works:
    parsed_rows.append(parse_work(work))

df = pd.DataFrame(parsed_rows)

print("Размер таблицы:", df.shape)
df.head()

Собрано: 100
Собрано: 200
Собрано: 300
Собрано: 400
Собрано: 500
Собрано: 600
Собрано: 700
Собрано: 800
Собрано: 900
Собрано: 1000
Собрано: 1100
Собрано: 1200
Размер таблицы: (1200, 13)


,work_id,title,publication_date,publication_year,type,cited_by_count,first_author,authors_count,primary_topic,is_open_access,oa_status,language,doi
0,https://openalex.org/W7125604336,"Policy Brief: Urban Informality, Social Welfar...",2026-11-17,2026,article,0,Selamawit Tesfaye,2,Urban and Rural Development Challenges,True,green,en,https://doi.org/10.5281/zenodo.18358234
1,https://openalex.org/W7130632997,Розумний будинок та інтернет речей: конспект л...,2026-11-17,2026,other,0,Ю.В. Левіков,1,IoT-based Smart Home Systems,False,closed,uk,
2,https://openalex.org/W7153772545,Gender-Responsive Budgeting in African Public ...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Gender Politics and Representation,True,green,en,https://doi.org/10.5281/zenodo.19540812
3,https://openalex.org/W7154052124,Technology and Procurement Reform: E-Procureme...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Public Procurement and Policy,True,green,en,https://doi.org/10.5281/zenodo.19548591
4,https://openalex.org/W7154064999,Platform Governance and Market Power: Regulati...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Diverse Legal and Medical Studies,True,green,en,https://doi.org/10.5281/zenodo.19550363


## 5. Проверка и подготовка данных

Перед сохранением в БД я проверила данные, `work_id` в OpenAlex уникальный, поэтому по нему можно искать дубли. Также проверила пропуски и перевела `publication_date` в формат даты

В текущей выборке получилось 1200 строк, 0 дублей и 0 пропусков. Значения, которых не было в исходных данных, еще на `parse_work()` заменили значениями вроде `unknown`

In [115]:
df["publication_date"] = pd.to_datetime(df["publication_date"], errors="coerce")

print("Размер таблицы:", df.shape)
print("Дубликаты:", df["work_id"].duplicated().sum())
print("\nПропуски:")
print(df.isna().sum())

Размер таблицы: (1200, 13)
Дубликаты: 0

Пропуски:
work_id             0
title               0
publication_date    0
publication_year    0
type                0
cited_by_count      0
first_author        0
authors_count       0
primary_topic       0
is_open_access      0
oa_status           0
language            0
doi                 0
dtype: int64


In [116]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   work_id           1200 non-null   object        
 1   title             1200 non-null   object        
 2   publication_date  1200 non-null   datetime64[ns]
 3   publication_year  1200 non-null   int64         
 4   type              1200 non-null   object        
 5   cited_by_count    1200 non-null   int64         
 6   first_author      1200 non-null   object        
 7   authors_count     1200 non-null   int64         
 8   primary_topic     1200 non-null   object        
 9   is_open_access    1200 non-null   bool          
 10  oa_status         1200 non-null   object        
 11  language          1200 non-null   object        
 12  doi               1200 non-null   object        
dtypes: bool(1), datetime64[ns](1), int64(3), object(8)
memory usage: 113.8+ KB


In [117]:
df.head()

,work_id,title,publication_date,publication_year,type,cited_by_count,first_author,authors_count,primary_topic,is_open_access,oa_status,language,doi
0,https://openalex.org/W7125604336,"Policy Brief: Urban Informality, Social Welfar...",2026-11-17,2026,article,0,Selamawit Tesfaye,2,Urban and Rural Development Challenges,True,green,en,https://doi.org/10.5281/zenodo.18358234
1,https://openalex.org/W7130632997,Розумний будинок та інтернет речей: конспект л...,2026-11-17,2026,other,0,Ю.В. Левіков,1,IoT-based Smart Home Systems,False,closed,uk,
2,https://openalex.org/W7153772545,Gender-Responsive Budgeting in African Public ...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Gender Politics and Representation,True,green,en,https://doi.org/10.5281/zenodo.19540812
3,https://openalex.org/W7154052124,Technology and Procurement Reform: E-Procureme...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Public Procurement and Policy,True,green,en,https://doi.org/10.5281/zenodo.19548591
4,https://openalex.org/W7154064999,Platform Governance and Market Power: Regulati...,2026-11-17,2026,article,0,Abraham Kuol Nyuon,1,Diverse Legal and Medical Studies,True,green,en,https://doi.org/10.5281/zenodo.19550363


## 6. Сохранение в SQLite

Для хранения данных я создала SQLite-базу `openalex.db`, а внутри нее таблицу `works`.

`to_sql()` переносит весь DataFrame в SQLite. Параметр `if_exists="replace"` здесь нужен для первоначального создания таблицы: если я заново запускаю всю лабораторную, старая таблица заменяется актуальной версией.

In [118]:
engine = create_engine("sqlite:///openalex.db")

In [119]:
df.to_sql("works", engine, if_exists="replace", index=False)

1200

После сохранения я отдельно проверяю количество строк в БД и вывожу первые пять записей, чтобы убедиться, что данные действительно записались, а не остались только в DataFrame


In [120]:
pd.read_sql_query(
    text("SELECT COUNT(*) AS records_count FROM works"),
    engine
)

,records_count
0,1200


In [121]:
pd.read_sql_query(
    text("SELECT * FROM works LIMIT 5"),
    engine
)

,work_id,title,publication_date,publication_year,type,cited_by_count,first_author,authors_count,primary_topic,is_open_access,oa_status,language,doi
0,https://openalex.org/W7125604336,"Policy Brief: Urban Informality, Social Welfar...",2026-11-17 00:00:00.000000,2026,article,0,Selamawit Tesfaye,2,Urban and Rural Development Challenges,1,green,en,https://doi.org/10.5281/zenodo.18358234
1,https://openalex.org/W7130632997,Розумний будинок та інтернет речей: конспект л...,2026-11-17 00:00:00.000000,2026,other,0,Ю.В. Левіков,1,IoT-based Smart Home Systems,0,closed,uk,
2,https://openalex.org/W7153772545,Gender-Responsive Budgeting in African Public ...,2026-11-17 00:00:00.000000,2026,article,0,Abraham Kuol Nyuon,1,Gender Politics and Representation,1,green,en,https://doi.org/10.5281/zenodo.19540812
3,https://openalex.org/W7154052124,Technology and Procurement Reform: E-Procureme...,2026-11-17 00:00:00.000000,2026,article,0,Abraham Kuol Nyuon,1,Public Procurement and Policy,1,green,en,https://doi.org/10.5281/zenodo.19548591
4,https://openalex.org/W7154064999,Platform Governance and Market Power: Regulati...,2026-11-17 00:00:00.000000,2026,article,0,Abraham Kuol Nyuon,1,Diverse Legal and Medical Studies,1,green,en,https://doi.org/10.5281/zenodo.19550363


## 7. Дозагрузка новых данных

Сдеали обновление базы. Я запрашиваю свежие публикации, беру их `work_id` и сравниваю с ID, которые уже лежат в SQLite. Если ID уже есть, то такую работу пропускаю, в ином случае добавляю строку через `append`


In [122]:
def build_dataframe(works):
    rows = []
    for work in works:
        rows.append(parse_work(work))

    new_df = pd.DataFrame(rows)
    new_df["publication_date"] = pd.to_datetime(new_df["publication_date"], errors="coerce")
    return new_df

In [123]:
def update_database(check_count=200):
    fresh_works = collect_works(check_count)
    fresh_df = build_dataframe(fresh_works)

    saved_ids_df = pd.read_sql_query(
        text("SELECT work_id FROM works"),
        engine
    )
    saved_ids = set(saved_ids_df["work_id"])

    rows_to_add = fresh_df[~fresh_df["work_id"].isin(saved_ids)]

    if len(rows_to_add) > 0:
        rows_to_add.to_sql("works", engine, if_exists="append", index=False)
        print("Добавлено новых записей:", len(rows_to_add))
    else:
        print("Новых записей нет")

In [124]:
update_database(200)

Собрано: 100
Собрано: 200
Новых записей нет


После дозагрузки еще раз проверяю число строк. Если свежих публикаций среди проверенных записей нет, размер базы не меняется

In [125]:
pd.read_sql_query(
    text("SELECT COUNT(*) AS records_count FROM works"),
    engine
)

,records_count
0,1200


# 8. SQL-запросы
Несколько запросов к SQLite. Запрос пишется на SQL, а результат считываю обратно в DataFrame через `pd.read_sql_query()` и подключение `SQLAlchemy`

### 1. Какие типы публикаций встречаются чаще

Группирую работы по `type` и считаю количество строк в каждой группе. В текущей выборке больше всего обычных статей (745)

In [126]:
sql1 = """
SELECT
    type,
    COUNT(*) AS works_count
FROM works
GROUP BY type
ORDER BY works_count DESC
"""

result_types = pd.read_sql_query(text(sql1), engine)
result_types.head(10)

,type,works_count
0,article,745
1,dataset,92
2,conference-paper,90
3,preprint,87
4,dissertation,25
5,peer-review,24
6,paratext,24
7,software,23
8,book,21
9,other,19


### 2. Среднее число цитирований по типам

Здесь кроме количества работ считаю `AVG(cited_by_count)`

(очень высокий средний показатель у редкого типа может быть связан буквально с несколькими работами, тк выборка не совсем большая)

In [127]:
sql2 = """
SELECT
    type,
    COUNT(*) AS works_count,
    AVG(cited_by_count) AS avg_citations
FROM works
GROUP BY type
ORDER BY avg_citations DESC
"""

result_citations_by_type = pd.read_sql_query(text(sql2), engine)
result_citations_by_type.head(10)

,type,works_count,avg_citations
0,paratext,24,71.625000
1,data-paper,3,1.000000
2,conference-abstract,2,0.500000
3,article,745,0.225503
4,conference-paper,90,0.211111
5,review,14,0.142857
6,other,19,0.105263
7,report,12,0.083333
8,software,23,0.043478
9,dataset,92,0.032609


### 3. Самые частые темы работ

Группирую публикации по `primary_topic` и вывожу 10 самых частых тем. Часть работ OpenAlex не отнес к основной теме

In [128]:
sql3 = """
SELECT
    primary_topic,
    COUNT(*) AS works_count
FROM works
GROUP BY primary_topic
ORDER BY works_count DESC
LIMIT 10
"""

result_topics = pd.read_sql_query(text(sql3), engine)
result_topics

,primary_topic,works_count
0,Не указана,48
1,Seismology and Earthquake Studies,28
2,"Galaxies: Formation, Evolution, Phenomena",17
3,3D Surveying and Cultural Heritage,17
4,Astrophysical Phenomena and Observations,16
5,earthquake and tectonic studies,15
6,Climate variability and models,15
7,Magnetic properties of thin films,13
8,Historical Geography and Geographical Thought,11
9,Meteorological Phenomena and Simulations,8


### 4. Среднее число цитирований по темам

Для тем считаю среднее число цитирований, но оставляю только те, где есть хотя бы 5 работ (`HAVING COUNT(*) >= 5`), добавила это условие, чтобы вывод не строился на одной теме

In [129]:
sql4 = """
SELECT
    primary_topic,
    COUNT(*) AS works_count,
    AVG(cited_by_count) AS avg_citations
FROM works
GROUP BY primary_topic
HAVING COUNT(*) >= 5
ORDER BY avg_citations DESC
LIMIT 10
"""

result_topic_citations = pd.read_sql_query(text(sql4), engine)
result_topic_citations

,primary_topic,works_count,avg_citations
0,Не указана,48,18.291667
1,Historical Geography and Geographical Thought,11,0.818182
2,Remote Sensing and LiDAR Applications,6,0.166667
3,earthquake and tectonic studies,15,0.066667
4,Climate variability and models,15,0.066667
5,3D Surveying and Cultural Heritage,17,0.058824
6,Translation Studies and Practices,6,0.000000
7,Seismology and Earthquake Studies,28,0.000000
8,Political Economy and Marxism,5,0.000000
9,Parallel Computing and Optimization Techniques,6,0.000000


### 5. Частые языки публикаций

Считаю количество работ по языкам. В моей выборке заметно преобладает английский язык 1011 публикаций из 1200.

In [130]:
sql5 = """
SELECT
    language,
    COUNT(*) AS works_count
FROM works
GROUP BY language
ORDER BY works_count DESC
LIMIT 10
"""

result_languages = pd.read_sql_query(text(sql5), engine)
result_languages

,language,works_count
0,en,1011
1,pt,56
2,fr,35
3,es,22
4,id,18
5,unknown,13
6,bg,10
7,ru,7
8,de,5
9,uz,4


### 6. Статусы Open Access

Показываем сколько публикаций относится к каждому статусу открытого доступа


In [131]:
sql6 = """
SELECT
    oa_status,
    COUNT(*) AS works_count
FROM works
GROUP BY oa_status
ORDER BY works_count DESC
"""

result_open_access = pd.read_sql_query(text(sql6), engine)
result_open_access

,oa_status,works_count
0,green,536
1,closed,270
2,diamond,162
3,gold,155
4,hybrid,75
5,bronze,2


In [132]:
from google.colab import files

files.download("openalex.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Вывод

В этой работе я собрала **1200 научных публикаций** из OpenAlex, привела ответ API к обычной таблице и проверила данные перед сохранением. После этого сохранила их в SQLite и сделала отдельную функцию для дозагрузки новых работ без дублей.

Для анализа базы я выполнила 6 агрегирующих SQL-запросов: посмотрела типы публикаций, цитируемость, темы, языки и статусы Open Access